**Hyperparameter Comparison Study**

For training on 1.5M timesteps (original trained model is evaluated at 1.5M timesteps).

Note that Learning Rate=3e-4, Value Coeff=0.5, and Clipping Param=0.2 are the parameters from the standard trained model (STND).

|   #   | Learning Rate| Value Coefficient | Clipping Parameter | Mean Reward | Median Reward | Success Rate |
|:-----:|:------------:|:-----------------:|:------------------:|:-----------:|:-------------:|:------------:|
|  LR1  |     1e-4     |        0.5        |         0.2        |    43.99    |     54.00     |     0.0%     |
| STND  |     3e-4     |        0.5        |         0.2        |   925.73    |    1000.00    |    85.0%     |
|  LR2  |     6e-4     |        0.5        |         0.2        |   687.74    |     869.00    |    44.0%     |
| VAL1  |     3e-4     |        0.2        |         0.2        |   941.60    |    1000.00    |    89.0%     |
| STND  |     3e-4     |        0.5        |         0.2        |   925.73    |    1000.00    |    85.0%     |
| VAL2  |     3e-4     |        0.8        |         0.2        |   873.21    |    1000.00    |    75.0%     |
| CLIP1 |     3e-4     |        0.5        |         0.1        |   428.56    |       4.50    |    42.0%     |
| STND  |     3e-4     |        0.5        |         0.2        |   925.73    |    1000.00    |    85.0%     |
| CLIP2 |     3e-4     |        0.5        |         0.5        |   915.09    |    1000.00    |    91.0%     |

From these results, it appears that training is quite sensitive to varying learning rate, as the reward decrease when the learning rate is both decreased and increased. A very small learning rate results in the agent learning too slowly. A larger learning rate causes training to become unstable. Additionally, it appears that a lower value coefficient results in higher rewards. The lower the value coefficient, the more that the model learns to selected better actions (versus predict correct values). The values for this environment are relatively high, so having a smaller value coefficient would help balance the training. Finally, a larger clipping parameter (to some extent) improves the reward. The larger the clipping parameter, the larger policy updates are allowed. Increasing the clipping parameter beyond what it was in this study may result in diminishing rewards.

Based on the results of this hyperparameter study, the best configuration would be Learning Rate=3e-4, Value Coeff=0.2, and Clipping Param=0.4.

**Ablation Study**\
Otherwise using Learning Rate=3e-4, Value Coeff=0.5, and Clipping Param=0.2. Trained on 1.5M timesteps.

| #  |                Ablation               | Mean Reward | Std Reward | Median Reward | Success Rate |
|:--:|:-------------------------------------:|:-----------:|:----------:|---------------|:------------:|
| A1 |               No entropy              |   1000.00   |    0.0     |    1000.00    |    100.0%    |
| A2 |              No LayerNorm             |    901.47   |   231.56   |    1000.00    |     76.0%    |
| A3 |                 No GAE                |    640.08   |   371.21   |     719.00    |     42.0%    |
|STND|                  None                 |    925.73   |   203.83   |    1000.00    |     85.0%    |

Removing the LayerNorm in the actor-critic architecture resulted in worse performance compared to the standard trained model, although the reward was not significantly less. This is expected as the normalization of the activations stabilizes training. From these results, it can be inferred that training was not very unstable, as the agent was still able to perform well. Removing the generalized advantage estimate resulted in significantly worse performance. From the ablation study plot, it can be seen that the reward jumped up quickly during training with no GAE, but then steadily plateaued at a lower reward. It is possible that early on, the critic learned quickly or happened to start close to correct, but later on began to stray due to the instable nature of the environment. 

Interestingly, removing entropy yielded much better results than the standard. As this problem (single inverted pendulum) is not very complex, exploration of new states may not be necessary. It may be more beneficial to leave out entropy altogether so that the agent chooses the best action every time.

**Updated Agent**\
Train agent using Learning Rate=3e-4, Value Coeff=0.2, Clipping Param=0.4, and no entropy.

| #   |                   Model                   | Mean Reward | Std Reward | Median Reward | Success Rate |
|:---:|:-----------------------------------------:|:-----------:|:----------:|---------------|:------------:|
|STND |         Original trained agent            |    925.73   |   203.83   |    1000.00    |     85.0%    |
|FINAL|Agent trained with updated hyperparamenters|    975.73   |   129.75   |    1000.00    |     95.0%    |

The updated model shows clear improvement over the original trained agent.

In [15]:
# Run this cell at the beginning of each session to set up the environment
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/cs372_final_project')

# First time: clone repo
# !git clone https://github.com/victoriaf55/cs372-final.git

# Every subsequent session: pull latest
!git -C cs372-final pull

# Install dependencies
!apt-get install -y xvfb ffmpeg > /dev/null 2>&1
!pip install gymnasium[mujoco] torch tensorboard pyvirtualdisplay > /dev/null 2>&1

# Verify installation
import gymnasium as gym

env = gym.make("InvertedPendulum-v5")
obs, info = env.reset()
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
env.close()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 9 (delta 5), reused 9 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 334.04 KiB | 214.00 KiB/s, done.
From https://github.com/victoriaf55/cs372-final
   e65daf4..6276dde  main       -> origin/main
Updating e65daf4..6276dde
Fast-forward
 README.md                               |   4 +--
 docs/Update100_Step409600_Reward265.mp4 | Bin 0 -> 38526 bytes
 docs/updated_comparison.png             | Bin 0 -> 189110 bytes
 notebooks/plots.ipynb                   |  59 ++++++++++++++++++++++----------
 notebooks/train.ipynb                   |  18 +++-------
 5 files changed, 47 insertions(+), 34 deletions(-)
 create mode 100644 docs/Update100_Step409600_Reward265.mp4
 create mode 100644 docs/updated_comparis

In [12]:
# Run training loop
import os
os.chdir('/content/drive/MyDrive/cs372_final_project/cs372-final/src')
!python train.py

2026-04-26 21:29:33.633666: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777238973.654851   71886 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777238973.662183   71886 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777238973.680903   71886 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777238973.680957   71886 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777238973.680962   71886 computation_placer.cc:177] computation placer alr

In [14]:
# Evaluate
import os
os.chdir('/content/drive/MyDrive/cs372_final_project/cs372-final')
!python src/evaluate.py --checkpoint /content/drive/MyDrive/cs372_final_project/checkpoints/FINAL_lr0.0003_valcoeff0.2_clip0.4.pt

Loaded checkpoint: /content/drive/MyDrive/cs372_final_project/checkpoints/FINAL_lr0.0003_valcoeff0.2_clip0.4.pt

EVALUATION RESULTS
Episodes:       100
Mean Reward:    975.73
Std Reward:     129.75
Min Reward:     111.00
Max Reward:     1000.00
Median Reward:  1000.00
Mean Ep Length: 975.78
Success Rate:   95.0%

